# Kafka Administration with the Python Admin API (Confluent Cloud)

What you will practise:

1. Basic Admin API concepts
2. List / create / describe topics
3. Consumer groups and lag
4. Cluster health-check scripting
5. Scanning topic data: text, JSON and Avro
6. Automating repetitive administration (declarative topic management)

Requirements: `pip install "confluent-kafka[avro,schemaregistry]" python-dotenv fastavro`, plus the `.env` file in `confluent-cloud/` with
`BOOTSTRAP_SERVRS`, `KEY`, `SECRET`, `SCHEMA_REGISTRY_ENDPOINT`, `SCHEMA_REGISTRY_KEY`, `SCHEMA_REGISTRY_SECRET`.
All topics created here use the fixed prefix `admin-nb-` and one partition, and the last section deletes them.

## 1. Setup: load `.env` and build the clients

In [1]:
import json, time
from pathlib import Path
from dotenv import dotenv_values

# The notebook may be started from confluent-cloud/ or from its parent folder.
env_file = next(p for p in (Path(".env"), Path("confluent-cloud/.env")) if p.exists())
env = dotenv_values(env_file)          # a dict; nothing is exported to os.environ
print("Loaded", env_file.resolve(), "->", sorted(env))

Loaded /home/aio/training/confluent-python/confluent-cloud/.env -> ['BOOTSTRAP_SERVRS', 'KEY', 'REST_ENDPOINT', 'SCHEMA_REGISTRY_ENDPOINT', 'SCHEMA_REGISTRY_KEY', 'SCHEMA_REGISTRY_SECRET', 'SECRET']


In [ ]:
from confluent_kafka.admin import AdminClient

kafka_conf = {
    "bootstrap.servers": env["BOOTSTRAP_SERVRS"].replace("SASL_SSL://", ""),
    "security.protocol": "SASL_SSL",
    "sasl.mechanism": "PLAIN",
    "sasl.username": env["KEY"],
    "sasl.password": env["SECRET"],
}
admin = AdminClient(kafka_conf)
print(admin)

## 2. Basic Admin API concepts

* `AdminClient` talks to the brokers' admin protocol; it is separate from `Producer` / `Consumer`.
* Every operation is **asynchronous**: it returns a `Future` (or a `dict` of futures, one per topic / group / resource).
  Call `.result()` to wait and to raise any error.
* Errors are `KafkaException`s carrying a `KafkaError` (`err.code()`, `err.name()`, `err.str()`), e.g. `TOPIC_ALREADY_EXISTS`, `TOPIC_AUTHORIZATION_FAILED`.
* On Confluent Cloud you need the right ACLs / role bindings; a cluster API key of a super user is assumed here.

The helper below waits for a dict of futures and reports per-item success or failure, and is reused throughout the notebook.

In [ ]:
from confluent_kafka import KafkaException

def wait(futures, what="op"):
    """Wait for {name: Future}; print OK/FAILED per item and return {name: result or exception}."""
    out = {}
    for name, f in futures.items():
        try:
            out[name] = f.result()
            print(f"  OK     {what} {name}")
        except KafkaException as e:
            out[name] = e
            print(f"  FAILED {what} {name}: {e.args[0].name()} - {e.args[0].str()}")
    return out

md_cluster = admin.describe_cluster().result()     # cluster id, controller, brokers
print("cluster id :", md_cluster.cluster_id)
print("brokers    :", [(n.id, n.host) for n in md_cluster.nodes])

## 3. List, create and describe topics

In [ ]:
# 3.1 List topics (cluster metadata). timeout is in seconds.
cluster_md = admin.list_topics(timeout=20)
print("cluster:", cluster_md.cluster_id, "| topics:", len(cluster_md.topics))
for name, t in sorted(cluster_md.topics.items()):
    print(f"{name:45} partitions={len(t.partitions)}")

In [ ]:
# 3.2 Create topics: one partition, replication factor 3 is what Confluent Cloud enforces.
from confluent_kafka.admin import NewTopic

TOPICS = {
    "admin-nb-text":  {"retention.ms": "86400000"},
    "admin-nb-json":  {"retention.ms": "86400000"},
    "admin-nb-avro":  {"retention.ms": "86400000"},
}
new_topics = [NewTopic(n, num_partitions=1, replication_factor=3, config=c) for n, c in TOPICS.items()]
wait(admin.create_topics(new_topics), "create")

In [ ]:
# 3.3 Describe topics: partitions, leader, replicas and in-sync replicas
from confluent_kafka.admin import ConfigResource

topic_md = admin.list_topics("admin-nb-text", timeout=20).topics["admin-nb-text"]
for pid, p in topic_md.partitions.items():
    print(f"partition {pid}: leader={p.leader} replicas={p.replicas} isr={p.isrs}")

# 3.4 Describe topic configuration
res = ConfigResource(ConfigResource.Type.TOPIC, "admin-nb-text")
configs = admin.describe_configs([res])[res].result()
for k in ("cleanup.policy", "retention.ms", "min.insync.replicas", "max.message.bytes"):
    c = configs[k]
    print(f"{k:22} = {c.value:12} (default={c.is_default}, read_only={c.is_read_only})")

In [ ]:
# 3.5 Change configuration (incremental: only the listed keys change)
from confluent_kafka.admin import ConfigEntry, AlterConfigOpType

res = ConfigResource(ConfigResource.Type.TOPIC, "admin-nb-text",
                     incremental_configs=[ConfigEntry("retention.ms", "172800000",
                                                       incremental_operation=AlterConfigOpType.SET)])
wait(admin.incremental_alter_configs([res]), "alter")
res = ConfigResource(ConfigResource.Type.TOPIC, "admin-nb-text")
print("retention.ms now:", admin.describe_configs([res])[res].result()["retention.ms"].value)

## 4. Produce sample data: text, JSON and Avro

In [ ]:
from confluent_kafka import Producer
producer = Producer(kafka_conf)

def delivery(err, msg):
    if err:
        print("delivery failed:", err)

# Plain text
for i in range(5):
    producer.produce("admin-nb-text", value=f"hello kafka {i}".encode(), key=f"k{i}", callback=delivery)

# JSON
for i in range(5):
    doc = {"order_id": f"order-{i}", "customer_id": f"customer-{i % 2}", "amount": 10.5 * (i + 1)}
    producer.produce("admin-nb-json", value=json.dumps(doc).encode(), key=doc["order_id"], callback=delivery)
producer.flush(30)
print("text + json produced")

In [ ]:
# Avro via Schema Registry (schema is registered automatically under subject admin-nb-avro-value)
from confluent_kafka.schema_registry import SchemaRegistryClient
from confluent_kafka.schema_registry.avro import AvroSerializer, AvroDeserializer
from confluent_kafka.serialization import SerializationContext, MessageField, StringSerializer

sr = SchemaRegistryClient({
    "url": env["SCHEMA_REGISTRY_ENDPOINT"],
    "basic.auth.user.info": f"{env['SCHEMA_REGISTRY_KEY']}:{env['SCHEMA_REGISTRY_SECRET']}",
})

ORDER_SCHEMA = json.dumps({
    "type": "record", "name": "Order", "namespace": "training",
    "fields": [
        {"name": "order_id", "type": "string"},
        {"name": "customer_id", "type": "string"},
        {"name": "amount", "type": "double"},
    ],
})
avro_ser = AvroSerializer(sr, ORDER_SCHEMA)
key_ser = StringSerializer()

for i in range(5):
    doc = {"order_id": f"order-{i}", "customer_id": f"customer-{i % 2}", "amount": 10.5 * (i + 1)}
    producer.produce("admin-nb-avro",
                     key=key_ser(doc["order_id"]),
                     value=avro_ser(doc, SerializationContext("admin-nb-avro", MessageField.VALUE)),
                     callback=delivery)
producer.flush(30)
print("avro produced; subjects:", [s for s in sr.get_subjects() if s.startswith("admin-nb-")])

## 5. Consumer groups and lag

*Lag* = log end offset (latest) - the group's committed offset, per partition.
First we create a group by consuming and committing a few messages, then we inspect it with the Admin API.

In [ ]:
from confluent_kafka import Consumer

def consume_some(group, topic, n):
    """Consume up to n messages and commit, to create / advance a consumer group."""
    c = Consumer({**kafka_conf, "group.id": group, "auto.offset.reset": "earliest", "enable.auto.commit": False})
    c.subscribe([topic])
    got, deadline = 0, time.time() + 30
    while got < n and time.time() < deadline:
        m = c.poll(1.0)
        if m is not None and not m.error():
            got += 1
    c.commit(asynchronous=False)
    c.close()
    return got

print("consumed", consume_some("admin-nb-group", "admin-nb-text", 3), "of 5 messages")

In [ ]:
from confluent_kafka import ConsumerGroupTopicPartitions, TopicPartition
from confluent_kafka.admin import OffsetSpec

# 5.1 List groups
groups = admin.list_consumer_groups().result()
for g in groups.valid:
    print(f"{g.group_id:40} state={g.state.name:10} type={g.type.name}")
if groups.errors:
    print("errors:", groups.errors)

In [ ]:
# 5.2 Describe a group: state, coordinator, members and assignments
desc = admin.describe_consumer_groups(["admin-nb-group"])["admin-nb-group"].result()
print("state:", desc.state.name, "| coordinator:", desc.coordinator.host, "| members:", len(desc.members))
for m in desc.members:
    print(" member", m.member_id, m.client_id, m.assignment.topic_partitions if m.assignment else None)

In [ ]:
# 5.3 Lag per partition = latest offset - committed offset
def group_lag(group):
    committed = admin.list_consumer_group_offsets([ConsumerGroupTopicPartitions(group)])[group].result()
    tps = [tp for tp in committed.topic_partitions]
    if not tps:
        return []
    latest = admin.list_offsets({TopicPartition(tp.topic, tp.partition): OffsetSpec.latest() for tp in tps})
    rows = []
    for tp in tps:
        end = latest[TopicPartition(tp.topic, tp.partition)].result().offset
        rows.append({"topic": tp.topic, "partition": tp.partition,
                     "committed": tp.offset, "end": end, "lag": end - tp.offset})
    return rows

for r in group_lag("admin-nb-group"):
    print(r)

In [ ]:
# 5.4 Reset the group to the beginning offsets (group must have no active members)
from confluent_kafka.admin import OffsetSpec
earliest = admin.list_offsets({TopicPartition("admin-nb-text", 0): OffsetSpec.earliest()})
start = earliest[TopicPartition("admin-nb-text", 0)].result().offset
wait(admin.alter_consumer_group_offsets(
        [ConsumerGroupTopicPartitions("admin-nb-group", [TopicPartition("admin-nb-text", 0, start)])]),
     "reset-offsets")
print(group_lag("admin-nb-group"))

## 6. Cluster health-check script

A health check is a function that returns a list of `(check, status, detail)` rows, so it can be printed in a notebook,
run from cron or a CI job, and turned into an exit code. Confluent Cloud manages brokers for you, so the checks focus on what a client can observe:
reachability, controller / broker count, under-replicated or leaderless partitions, and consumer lag.

In [ ]:
def health_check(topic_prefix="", lag_threshold=1000, expected_rf=3):
    rows = []
    def add(check, ok, detail): rows.append((check, "PASS" if ok else "FAIL", detail))

    # 1. connectivity + brokers
    try:
        c = admin.describe_cluster(request_timeout=15).result()
        add("cluster reachable", True, f"id={c.cluster_id} brokers={len(c.nodes)}")
        add("controller present", c.controller is not None, f"controller={getattr(c.controller, 'id', None)}")
    except Exception as e:
        add("cluster reachable", False, str(e))
        return rows

    # 2. topics: leaderless / under-replicated partitions
    md = admin.list_topics(timeout=20)
    bad = []
    for name, t in md.topics.items():
        if not name.startswith(topic_prefix):
            continue
        for pid, p in t.partitions.items():
            if p.leader < 0:
                bad.append(f"{name}[{pid}] no leader")
            elif len(p.isrs) < len(p.replicas):
                bad.append(f"{name}[{pid}] isr={len(p.isrs)}/{len(p.replicas)}")
            elif len(p.replicas) < expected_rf:
                bad.append(f"{name}[{pid}] rf={len(p.replicas)}")
    add("partition health", not bad, "; ".join(bad) or f"{sum(len(t.partitions) for t in md.topics.values())} partitions OK")

    # 3. consumer groups: state and lag
    gl = admin.list_consumer_groups().result().valid
    for g in gl:
        if not g.group_id.startswith(topic_prefix.split("-")[0] if topic_prefix else ""):
            continue
        try:
            lag = sum(r["lag"] for r in group_lag(g.group_id))
        except Exception as e:
            add(f"group {g.group_id}", False, f"cannot read offsets: {e}")
            continue
        add(f"group {g.group_id}", lag <= lag_threshold, f"state={g.state.name} lag={lag}")

    # 4. Schema Registry
    try:
        add("schema registry", True, f"{len(sr.get_subjects())} subjects")
    except Exception as e:
        add("schema registry", False, str(e))
    return rows

def print_report(rows):
    for check, status, detail in rows:
        print(f"{status:5} {check:28} {detail}")
    return all(s == "PASS" for _, s, _ in rows)

healthy = print_report(health_check(topic_prefix="admin-nb-"))
print("\nOVERALL:", "HEALTHY" if healthy else "UNHEALTHY")
# In a script: sys.exit(0 if healthy else 1)

## 7. Scan topic data: text, JSON and Avro

`scan_topic` reads a topic **without a consumer group** (`assign` + explicit offsets, no commits), so scanning never disturbs real consumers.
It detects the format of each message: Confluent wire format for Avro (magic byte `0x00` + 4-byte schema id), JSON, or plain text.

In [ ]:
from confluent_kafka import OFFSET_BEGINNING

avro_deser = AvroDeserializer(sr)       # reads the writer schema from the registry using the id in the message

def decode(msg):
    """Return (format, value) for one message."""
    raw = msg.value()
    if raw is None:
        return "tombstone", None
    if raw[:1] == b"\x00" and len(raw) > 5:
        try:
            return "avro", avro_deser(raw, SerializationContext(msg.topic(), MessageField.VALUE))
        except Exception as e:
            return "avro-error", str(e)
    text = raw.decode("utf-8", errors="replace")
    try:
        return "json", json.loads(text)
    except ValueError:
        return "text", text

def scan_topic(topic, limit=100, from_end=False):
    """Scan up to `limit` messages per partition. Yields dicts with metadata + decoded value."""
    md = admin.list_topics(topic, timeout=20).topics[topic]
    c = Consumer({**kafka_conf, "group.id": "admin-nb-scanner", "enable.auto.commit": False,
                  "enable.partition.eof": True})
    try:
        tps = []
        for pid in md.partitions:
            lo, hi = c.get_watermark_offsets(TopicPartition(topic, pid), timeout=20)
            start = max(lo, hi - limit) if from_end else lo
            tps.append(TopicPartition(topic, pid, start))
        c.assign(tps)
        done, count = set(), {p.partition: 0 for p in tps}
        while len(done) < len(tps):
            m = c.poll(5.0)
            if m is None:
                break
            if m.error():
                if m.error().code() == -191:           # _PARTITION_EOF
                    done.add(m.partition())
                continue
            fmt, val = decode(m)
            yield {"partition": m.partition(), "offset": m.offset(), "ts": m.timestamp()[1],
                   "key": m.key().decode() if m.key() else None, "format": fmt, "value": val}
            count[m.partition()] += 1
            if count[m.partition()] >= limit:
                done.add(m.partition())
    finally:
        c.close()

for t in ("admin-nb-text", "admin-nb-json", "admin-nb-avro"):
    print("==", t)
    for r in scan_topic(t, limit=3):
        print(f"  p{r['partition']}@{r['offset']} key={r['key']} [{r['format']}] {r['value']}")

In [ ]:
# Scan a topic and summarise: message count, formats seen, and a simple data-quality rule.
from collections import Counter

def summarise(topic, limit=10_000):
    formats, n, bad = Counter(), 0, []
    for r in scan_topic(topic, limit=limit):
        n += 1
        formats[r["format"]] += 1
        if r["format"] in ("json", "avro") and not (isinstance(r["value"], dict) and r["value"].get("amount", 0) > 0):
            bad.append((r["partition"], r["offset"]))
    return {"topic": topic, "messages": n, "formats": dict(formats), "bad_records": bad}

for t in ("admin-nb-json", "admin-nb-avro"):
    print(summarise(t))

In [ ]:
# Search for a value across a topic (e.g. find every record of one customer)
hits = [r for r in scan_topic("admin-nb-json") if isinstance(r["value"], dict) and r["value"]["customer_id"] == "customer-1"]
print(len(hits), "matches:", [h["offset"] for h in hits])

## 8. Automating cluster administration

### 8.1 Declarative topic management

Describe the topics you *want* in a dict and let a `reconcile` function create what is missing and fix drifted configuration.
`dry_run=True` shows the plan without changing anything. This is idempotent: running it twice changes nothing the second time.

In [ ]:
DESIRED = {
    "admin-nb-text":   {"partitions": 1, "config": {"retention.ms": "172800000"}},
    "admin-nb-json":   {"partitions": 1, "config": {"retention.ms": "86400000"}},
    "admin-nb-avro":   {"partitions": 1, "config": {"retention.ms": "86400000"}},
    "admin-nb-events": {"partitions": 1, "config": {"retention.ms": "3600000", "cleanup.policy": "delete"}},
}

def reconcile(desired, dry_run=True):
    existing = admin.list_topics(timeout=20).topics
    plan = []
    to_create = [n for n in desired if n not in existing]
    for n in to_create:
        plan.append(("create", n, desired[n]["config"]))

    alters = []
    for n, spec in desired.items():
        if n in existing:
            r = ConfigResource(ConfigResource.Type.TOPIC, n)
            current = admin.describe_configs([r])[r].result()
            diff = {k: v for k, v in spec["config"].items() if current[k].value != v}
            have = len(existing[n].partitions)
            if have != spec["partitions"]:
                plan.append(("WARN", n, f"partitions {have} != {spec['partitions']} (cannot shrink)"))
            if diff:
                plan.append(("alter", n, diff))
                alters.append(ConfigResource(ConfigResource.Type.TOPIC, n, incremental_configs=[
                    ConfigEntry(k, v, incremental_operation=AlterConfigOpType.SET) for k, v in diff.items()]))

    for p in plan:
        print("PLAN", p)
    if not plan:
        print("nothing to do - cluster matches the desired state")
    if dry_run:
        return plan
    if to_create:
        wait(admin.create_topics([NewTopic(n, desired[n]["partitions"], 3, config=desired[n]["config"]) for n in to_create]), "create")
    if alters:
        wait(admin.incremental_alter_configs(alters), "alter")
    return plan

reconcile(DESIRED, dry_run=True)       # review the plan

In [ ]:
reconcile(DESIRED, dry_run=False)      # apply
print("--- second run (idempotent) ---")
reconcile(DESIRED, dry_run=True);

### 8.2 Housekeeping: find idle groups and empty topics

Typical repetitive jobs: report groups in `EMPTY` state, topics with no data, and topics outside a naming convention.

In [ ]:
def housekeeping_report(prefix="admin-nb-"):
    print("Empty consumer groups:")
    for g in admin.list_consumer_groups().result().valid:
        if g.state.name == "EMPTY":
            print("  ", g.group_id)

    print("Topics without data (or not following the naming convention):")
    c = Consumer({**kafka_conf, "group.id": "admin-nb-scanner"})
    try:
        for name in sorted(admin.list_topics(timeout=20).topics):
            if name.startswith("_") or name.startswith("confluent"):
                continue
            parts = admin.list_topics(name, timeout=20).topics[name].partitions
            n = sum(hi - lo for lo, hi in (c.get_watermark_offsets(TopicPartition(name, p), timeout=20) for p in parts))
            flags = []
            if n == 0: flags.append("EMPTY")
            if not name.startswith(prefix): flags.append("outside convention")
            if flags:
                print(f"   {name:45} messages={n} {', '.join(flags)}")
    finally:
        c.close()

housekeeping_report()

### 8.3 Run the health check on a schedule (script form)

In [ ]:
# Save the health check as a standalone script that exits non-zero when unhealthy, e.g. for cron:
#   */5 * * * * /path/to/venv/bin/python kafka_health.py >> /var/log/kafka_health.log 2>&1
# Here we only simulate three consecutive runs.
for run in range(3):
    ok = all(s == "PASS" for _, s, _ in health_check("admin-nb-"))
    print(time.strftime("%H:%M:%S"), "HEALTHY" if ok else "UNHEALTHY")
    time.sleep(2)

## 9. Clean up

In [ ]:
wait(admin.delete_consumer_groups(["admin-nb-group", "admin-nb-scanner"]), "delete-group")
wait(admin.delete_topics([t for t in admin.list_topics(timeout=20).topics if t.startswith("admin-nb-")]), "delete-topic")
for s in sr.get_subjects():
    if s.startswith("admin-nb-"):
        sr.delete_subject(s); sr.delete_subject(s, permanent=True); print("  OK     deleted subject", s)

## Review questions

1. Why does every `AdminClient` call return a future? What happens if you never call `.result()`?
2. Why does the scanner use `assign()` rather than `subscribe()`?
3. How is lag computed, and why can a group show lag of 0 on a topic that is still receiving data?
4. Why is `reconcile` safe to run repeatedly? What can it *not* fix (partition count)?